# Lecture 03c: Comparing Two Variables: the code

Every code cell from the slides, in order. Run them top to bottom.


## Covariance: do they move together?

In [ ]:
from IPython.display import YouTubeVideo
# until about 6 minutes
YouTubeVideo('qtaqvPAeEJY')

## In action: study hours vs. exam score

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df1 = pd.DataFrame({
    'study_hours': [1, 2, 3, 4, 5, 6, 7, 8],
    'exam_score':  [50, 52, 55, 58, 63, 67, 75, 82]
})

# 1. Compute Pearson correlation
corr_value = df1['study_hours'].corr(df1['exam_score'], method='pearson')
print(f"Pearson correlation: {corr_value:.3f}")

# 2. Compute covariance
cov_value = df1['study_hours'].cov(df1['exam_score'])
print(f"Covariance: {cov_value:.3f}")

## A correlation matrix: every pair at once

In [ ]:
# Create a simple dataset
df = pd.DataFrame({
    'study_hours': [2, 3, 4, 5, 6, 7],
    'exam_score':  [50, 55, 65, 70, 75, 85],
    'sleep_hours': [8, 7, 6, 6, 5, 5]
})

# Compute the correlation matrix
corr_matrix = df.corr(method='pearson')
print(corr_matrix)

## Study hours vs. exam score, as a scatter plot

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Simple scatter plot
sns.scatterplot(
    data=df1,
    x='study_hours',
    y='exam_score',
    s=70,
    color='steelblue',
    edgecolor='black'
)

plt.title(f"Study Hours vs Exam Score (r = {corr_value:.2f})", fontsize=13)
plt.xlabel("Hours Studied")
plt.ylabel("Exam Score")
plt.show()

## A line plot: 30 days of temperature

In [ ]:
import numpy as np
# Simulated data: temperature over 30 days
np.random.seed(0)
days = np.arange(1, 31)
temperature = 20 + 5 * np.sin(days / 4) + np.random.normal(0, 1, 30)
df = pd.DataFrame({'day': days, 'temperature': temperature})

sns.lineplot(data=df, x='day', y='temperature', marker='o', linewidth=2, color='steelblue')
plt.title('Daily Temperature Trend', fontsize=14, fontweight='bold')
plt.xlabel('Day')
plt.ylabel('Temperature (°C)')
plt.grid(alpha=0.3)
plt.show()

## Files

In [ ]:
import pandas as pd

excel_dataframe = pd.read_excel('figs/03_csv_toy.xlsx') # read_csv for .csv files
# (read_excel needs the openpyxl package: pip install openpyxl)
print(excel_dataframe)

## Reminder: the libraries we use

In [ ]:
import numpy as np                  # arrays and math
import pandas as pd                 # DataFrames
import matplotlib.pyplot as plt     # plots
import seaborn as sns               # statistical plots, built on Matplotlib

## Live data

In [ ]:
sheet_id = "1E6Gvi4QBLIRWRZPy7gE5qjxMEZYuDVKSDmDP3ICejrc"
url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv"
df = pd.read_csv(url)
df.head(3).T     # .T flips rows and columns, so the long question names fit

## Shorten the column names

In [ ]:
old_column_names = df.columns
print(old_column_names)

In [ ]:
new_column_names = {
    old_column_names[0]: old_column_names[0],
    old_column_names[1]: 'shoe',
    old_column_names[2]: 'random',
    old_column_names[3]: 'height',
    old_column_names[4]: 'color'
}
df = df.rename(columns=new_column_names)

## Turn text answers into numbers

In [ ]:
df['height'] = pd.to_numeric(df['height'].astype(str).str[:3], errors='coerce')   # first 3 characters
df['shoe']   = pd.to_numeric(df['shoe'],   errors='coerce')
df['random'] = pd.to_numeric(df['random'], errors='coerce')
df.head()

## Drop missing values and outliers (the 1.5 × IQR rule)

In [ ]:
df = df.dropna()

numeric_cols = df.select_dtypes(include='number').columns

for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    df = df[(df[col] >= lower_bound) & (df[col] <= upper_bound)]

print(df.shape)
df.head()

## Let's analyze: `df.describe()`

In [ ]:
df.describe()

## Exercise: univariate or bivariate?

In [ ]:
df.groupby('color')['height'].mean()

## Every column at once: `df.hist()`

In [ ]:
df.hist()

## Box plots: height and shoe size

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df[["height", "shoe"]], palette="Set2")
plt.title("Box Plots of Height and Shoe Size", fontsize=14, fontweight="bold")
plt.ylabel("Value")
plt.show()

## Height by favorite color

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x="color", y="height", hue="color", palette="Set3", legend=False)
plt.title("Height by Favorite Color", fontsize=14, fontweight="bold")
plt.xlabel("")
plt.ylabel("Height")
plt.show()

## Bar or pie? Which is more useful?

In [ ]:
# clean the names first, so 'blue', 'Blue ' and 'BLUE' count as one color
df['color'] = df['color'].str.strip().str.title()

plt.figure(figsize=(6,4))
sns.countplot(x='color', hue='color', data=df, palette=list(df['color'].unique()), legend=False)
plt.title('Favorite Color Choices', fontsize=14, fontweight='bold')
plt.xlabel('Color')
plt.ylabel('Count')
plt.show()

In [ ]:
color_counts = df['color'].value_counts()   # already cleaned

plt.figure(figsize=(6,6))
plt.pie(
    color_counts,
    labels=color_counts.index,
    autopct='%1.0f%%',
    colors=color_counts.index,   # use actual color names for slices
    startangle=90,
    textprops={'fontsize': 12, 'weight': 'bold'}
)
plt.title('Favorite Color Distribution', fontsize=14, fontweight='bold')
plt.show()

## Bivariate: height vs. shoe size

In [ ]:
plt.scatter(df['height'], df['shoe'])
plt.xlabel('Height [centimeters]')
plt.ylabel('Shoe Size [unisex]')
plt.grid()

## Every pair at once: a scatter matrix

In [ ]:
numeric_df = df.select_dtypes(include=[np.number])

# Compute correlations
correlations = numeric_df.corr()

n = len(numeric_df.columns)
plt.figure(figsize=(4*n, 4*n))

for i, c0 in enumerate(numeric_df.columns):
    for j, c1 in enumerate(numeric_df.columns):
        plt.subplot(n, n, i * n + j + 1)
        
        if i == j:
            # plot variable vs itself as a line
            plt.scatter(numeric_df[c0], numeric_df[c0])
        else:
            # scatter for different pairs
            plt.scatter(numeric_df[c0], numeric_df[c1], alpha=0.7)
        
        plt.xlabel(c0)
        plt.ylabel(c1)
        plt.title(f"corr={correlations.loc[c0, c1]:.4f}")
        plt.grid(alpha=0.3)

plt.suptitle("Pairwise Scatter Plots of Numeric Variables", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## Is this a good line plot?

In [ ]:
df['Timestamp'] = pd.to_datetime(df['Timestamp'])
df_sorted = df.sort_values('Timestamp')

plt.figure(figsize=(8, 4))
plt.plot(df_sorted['Timestamp'], df_sorted['height'], marker='o', linestyle='-', color='steelblue')
plt.title("Height over Time (by Submission Order)", fontsize=14, fontweight='bold')
plt.xlabel("Submission Time")
plt.ylabel("Height (cm)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()